# Smart Facility Platform — Local AI Lab

This standalone lab runs the same two-stage intelligence path as the dashboard, without Telegram:

1. **Local vision:** `qwen3-vl:30b` inspects the selected image through Ollama and returns structured findings plus normalized bounding boxes.
2. **Local agent:** NVIDIA NeMo Agent Toolkit runs a tool-calling workflow backed by `qwen3:32b` through Ollama. The agent must call the local `facility_sop` tool before producing its response.
3. **Evidence:** OpenCV draws the returned annotations and saves the result under `lab_outputs/`.

Both models execute on this machine. NeMo Agent Toolkit is the orchestration framework, not a hosted model.

## 0. Start the local services

From the repository root, use two terminals:

```bash
./scripts/run-nemo-agent.sh
```

Ollama should already be running as a local service. Confirm the models with:

```bash
ollama list
```

To launch this notebook in JupyterLab:

```bash
source .app-venv/bin/activate
python -m pip install -r requirements-lab.txt
jupyter lab
```

Run the remaining cells in order.

In [ ]:
from pathlib import Path
import base64
import json
import re
import urllib.error
import urllib.request

import cv2
from IPython.display import Image as NotebookImage, display

# Change only this value to select an image from lab_images/.
IMAGE_NAME = "your-image.jpg"

IMAGE_DIR = Path("lab_images")
OUTPUT_DIR = Path("lab_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

OLLAMA_URL = "http://127.0.0.1:11434"
VISION_MODEL = "qwen3-vl:30b"
NEMO_URL = "http://127.0.0.1:8010/v1"
NEMO_WORKFLOW = "smart-facility-agent"

print(f"Vision: {VISION_MODEL} via local Ollama")
print("Agent reasoning: qwen3:32b via local Ollama, orchestrated by NVIDIA NeMo Agent Toolkit")

## 1. Verify the local runtimes

This cell checks Ollama and the NeMo Agent Toolkit server before any inference. No cloud API is used.

In [ ]:
def get_json(url, timeout=10):
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))

ollama_version = get_json(f"{OLLAMA_URL}/api/version")
nemo_health = get_json(f"{NEMO_URL.rsplit('/v1', 1)[0]}/health")
available_images = sorted(p.name for p in IMAGE_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})

print("Ollama:", ollama_version)
print("NeMo Agent Toolkit:", nemo_health)
print("Available images:", available_images or "Add an image to lab_images/")

## 2. Select and preview the image

Set `IMAGE_NAME` in the configuration cell to a filename listed above.

In [ ]:
image_path = IMAGE_DIR / IMAGE_NAME
if not image_path.is_file():
    raise FileNotFoundError(f"Place {IMAGE_NAME!r} in {IMAGE_DIR.resolve()} and run this cell again.")

frame = cv2.imread(str(image_path))
if frame is None:
    raise ValueError(f"OpenCV could not decode {image_path}")

height, width = frame.shape[:2]
print(f"Selected: {image_path} ({width} × {height})")
display(NotebookImage(filename=str(image_path)))

## 3. Run local multimodal vision

`qwen3-vl:30b` receives the image bytes and a strict JSON schema. Bounding boxes use a 0–1000 coordinate space so they can be mapped onto images of any size.

In [ ]:
VISION_PROMPT = """Inspect this facility image for visible safety, access, parking, fire-exit, signage, housekeeping, or obstruction violations.
Return one JSON object only with these fields:
- violation: boolean
- category: short snake_case label
- severity: low, medium, high, or critical
- confidence: number from 0 to 1
- summary: precise description of what is wrong and where
- evidence: array of short visible observations
- visible_objects: array of relevant object labels
- recommended_action: immediate practical action
- annotations: array of objects with label, confidence, and box [x1,y1,x2,y2]
  where coordinates are integers normalized from 0 to 1000.
Do not invent hidden facts. If there is no clear violation, set violation false and annotations to an empty array."""

mime = "image/png" if image_path.suffix.lower() == ".png" else "image/jpeg"
image_b64 = base64.b64encode(image_path.read_bytes()).decode("ascii")
vision_payload = {
    "model": VISION_MODEL,
    "stream": False,
    "format": "json",
    "options": {"temperature": 0},
    "messages": [{
        "role": "user",
        "content": VISION_PROMPT,
        "images": [image_b64],
    }],
}

request = urllib.request.Request(
    f"{OLLAMA_URL}/api/chat",
    data=json.dumps(vision_payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(request, timeout=300) as response:
    vision_response = json.loads(response.read().decode("utf-8"))

raw_vision = vision_response["message"]["content"]
print(raw_vision)

In [ ]:
def parse_json_object(text):
    cleaned = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    cleaned = cleaned.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start < 0 or end < start:
        raise ValueError(f"No JSON object found in model response: {cleaned[:300]}")
    return json.loads(cleaned[start:end + 1])

vision = parse_json_object(raw_vision)
required = {"violation", "category", "severity", "confidence", "summary", "evidence", "visible_objects", "recommended_action", "annotations"}
missing = required.difference(vision)
if missing:
    raise ValueError(f"Vision response is missing: {sorted(missing)}")

print(json.dumps(vision, indent=2))

## 4. Draw and save evidence annotations

This is deterministic post-processing: OpenCV maps the model's normalized boxes to pixels and adds a high-contrast incident banner.

In [ ]:
def annotate_image(source, finding):
    canvas = source.copy()
    h, w = canvas.shape[:2]
    violation = bool(finding.get("violation"))
    color = (72, 72, 255) if violation else (127, 226, 73)

    for item in finding.get("annotations", []):
        box = item.get("box", [])
        if len(box) != 4:
            continue
        x1, y1, x2, y2 = [int(float(v)) for v in box]
        x1, x2 = sorted((max(0, min(w - 1, x1 * w // 1000)), max(0, min(w - 1, x2 * w // 1000))))
        y1, y2 = sorted((max(0, min(h - 1, y1 * h // 1000)), max(0, min(h - 1, y2 * h // 1000))))
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, max(2, round(min(w, h) / 250)))
        label = str(item.get("label", finding.get("category", "finding"))).replace("_", " ").upper()
        confidence = item.get("confidence")
        if isinstance(confidence, (int, float)):
            label += f"  {confidence:.0%}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
        top = max(0, y1 - th - 12)
        cv2.rectangle(canvas, (x1, top), (min(w - 1, x1 + tw + 12), y1), color, -1)
        cv2.putText(canvas, label, (x1 + 6, y1 - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (8, 15, 12), 2, cv2.LINE_AA)

    banner = "VIOLATION: " + str(finding.get("category", "unknown")).replace("_", " ").upper() if violation else "NO CLEAR VIOLATION"
    cv2.rectangle(canvas, (0, 0), (w, 54), color, -1)
    cv2.putText(canvas, banner, (18, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (8, 15, 12), 2, cv2.LINE_AA)
    return canvas

annotated = annotate_image(frame, vision)
annotated_path = OUTPUT_DIR / f"{image_path.stem}-annotated.jpg"
if not cv2.imwrite(str(annotated_path), annotated):
    raise OSError(f"Could not write {annotated_path}")

print("Saved:", annotated_path.resolve())
display(NotebookImage(filename=str(annotated_path)))

## 5. Send the finding through NVIDIA NeMo Agent Toolkit

The request below goes to the local NeMo server, not directly to the text model. The configured `tool_calling_agent` uses local `qwen3:32b`, calls the repository's `facility_sop` tool to retrieve the relevant operating procedure, and then returns grounded response guidance.

In [ ]:
incident_context = {
    "facility": "Notebook lab",
    "zone": "Selected image",
    "event_type": vision["category"],
    "severity": vision["severity"],
    "object_type": ", ".join(vision["visible_objects"]) or "unknown",
    "confidence": vision["confidence"],
    "visual_summary": vision["summary"],
    "visual_recommendation": vision["recommended_action"],
}

agent_payload = {
    "model": NEMO_WORKFLOW,
    "stream": False,
    "temperature": 0,
    "messages": [
        {
            "role": "system",
            "content": "You are a facility safety response agent. You must call facility_sop for the supplied event_type. Return one JSON object with exactly summary, recommended_action, and sop_title. Use only visible facts and retrieved SOP guidance."
        },
        {"role": "user", "content": json.dumps(incident_context)},
    ],
}

request = urllib.request.Request(
    f"{NEMO_URL}/chat/completions",
    data=json.dumps(agent_payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(request, timeout=300) as response:
    agent_response = json.loads(response.read().decode("utf-8"))

raw_agent = agent_response["choices"][0]["message"]["content"]
agent_guidance = parse_json_object(raw_agent)
print(json.dumps(agent_guidance, indent=2))

## 6. Review the combined local result

The final object records which component produced each part. Telegram is intentionally absent from this lab.

In [ ]:
lab_result = {
    "input_image": str(image_path),
    "annotated_image": str(annotated_path),
    "vision": {
        "model": VISION_MODEL,
        "runtime": "local Ollama",
        "finding": vision,
    },
    "agent": {
        "model": "qwen3:32b",
        "runtime": "local Ollama",
        "framework": "NVIDIA NeMo Agent Toolkit tool_calling_agent",
        "tool": "facility_sop",
        "guidance": agent_guidance,
    },
}

result_path = OUTPUT_DIR / f"{image_path.stem}-result.json"
result_path.write_text(json.dumps(lab_result, indent=2), encoding="utf-8")
print(json.dumps(lab_result, indent=2))
print("Result saved:", result_path.resolve())